# El Niño 2026 en la costa norte del Perú con NISAR (banda L)
### Procesamiento de escenas GCOV, GUNW y SME2 para Tumbes y Piura, cruzado con noticias

El código vive en el paquete `nisar_cn/` (un archivo por etapa) y este notebook lo importa y explica. Sigue la estructura de `pipeline/fetch_nisar.py` del proyecto *nsac-project-main* del equipo: búsqueda, geometría única, lectura de la ventana de la zona, clasificación y manifiesto para la web. Sobre esa base agrega:

1. **Zonas y eventos referenciados.** Cada zona entra porque una noticia abierta y citada documenta un daño en una fecha concreta.
2. **Pares antes/después automáticos** para cada evento, con la misma geometría de adquisición.
3. **La física que pide el reto:** sensibilidad a la **rugosidad** y a la **constante dieléctrica** con el modelo de Dubois (HH) y la relación de Topp (ε ↔ humedad). Incluye un mapa de Δε por detección de cambios.
4. **Indicadores adicionales:** Δσ⁰ (dB), cociente HH/HV, coherencia (GUNW) y humedad del suelo (SME2).
5. **Tabla para el storytelling** y un manifiesto compatible con la app web del equipo.

**Modos de ejecución**
- `MODO = "DEMO"`: no descarga nada. Usa el catálogo real de escenas que consulté en el CMR el 14 nov 2026 y genera archivos HDF5 **sintéticos** con la misma estructura que un GCOV, para probar el código de punta a punta. **Ninguna cifra del modo DEMO es un resultado científico.**
- `MODO = "REAL"`: consulta el CMR en vivo, descarga las escenas con tu token de Earthdata y procesa los datos reales.

**Convención de fuentes (regla del proyecto):** cada afirmación lleva una **caja Fuente** con estatus:
**[V]** verificado en una fuente abierta en la sesión de preparación (14 nov 2026) · **[A]** deducción propia a partir de una fuente citada (se muestra el razonamiento) · **[?]** hueco: falta referencia y se pide al equipo.

## 0. Referencias

| ID | Referencia | Qué respalda | Estatus |
|---|---|---|---|
| R1 | NASA Space Apps Challenge 2026, reto *Dancing with the SARs*, pestaña *Details*, sección *Background*. https://www.spaceappschallenge.org/2026/challenges/dancing-with-the-sars/?tab=details | El radar es más sensible a la rugosidad y a la constante dieléctrica; el agua tiene constante dieléctrica alta | [V] |
| R2 | ASF DAAC, *NISAR Data User Guide*, página «Geocoded Polarimetric Covariance (GCOV)». https://nisar-docs.asf.alaska.edu/gcov/ (cita a Shiroma, G. H. X. (2026), *Level-2 Geocoded Polarimetric Covariance*, JPL D-102274 Rev. F) | Rutas HDF5 (`HHHH`, `HVHV`, `numberOfLooks`, `mask`, `rtcGammaToSigmaFactor`), γ⁰ con RTC, píxel de 10/20 m según ancho de banda, convención de nombres (`MODE`, `POLE`, `CRID`) | [V] |
| R3 | ASF DAAC, *NISAR Data User Guide*, «Data Products Overview». https://nisar-docs.asf.alaska.edu/products-overview/ | Madurez BETA/PROVISIONAL/VALIDATED, precaución al mezclarlas, VALIDATED desde 21 oct 2026, SME2 a 200 m con 3 algoritmos (DSG, PMI, TSR) | [V] |
| R4 | ASF DAAC, *NISAR Data User Guide*, «GUNW». https://nisar-docs.asf.alaska.edu/gunw/ | GUNW: solo co-pol (HH o VV), pares de 12 días, coherencia a 20 y 80 m | [V] |
| R5 | ASF DAAC, *NISAR Data User Guide*, «About NISAR», tabla 1. https://nisar-docs.asf.alaska.edu/nisar-intro/ | Frecuencia central de la banda A según el modo (40+5 MHz → 1239 MHz; 20+5 MHz → 1229 MHz) | [V] |
| R6 | NASA Earthdata, «Synthetic Aperture Radar (SAR)». https://www.earthdata.nasa.gov/learn/earth-observation-data-basics/sar | Mecanismos: superficie rugosa (sensible a VV), volumen (sensible a HV/VH), rebote doble por troncos o vegetación inundada (sensible a HH); la banda L (~23 cm) penetra el dosel | [V] |
| R7 | Meyer, F. J. (2023). *GEOS 657 Microwave Remote Sensing*, Lecture 17 «Surface Water Mapping from SAR», University of Alaska Fairbanks, diapositivas 5, 6, 7 y 10. https://radar.community.uaf.edu/wp-content/uploads/sites/667/2023/04/2023-Lectures17-SurfaceWaterMapping-handouts.pdf | El agua en calma da reflexión especular y retrodispersión baja; la tierra es más rugosa; rebote doble bajo vegetación inundada; σ⁰ sube con la humedad del suelo en terreno abierto | [V] |
| R8 | Dubois, P., van Zyl, J., Engman, T. (1995). «Measuring soil moisture with imaging radars». *IEEE Trans. Geosci. Remote Sens.* 33(4), 915–926. doi:10.1109/36.406677 | Modelo semiempírico σ⁰_HH(ε, ks, θ, λ) y límites de validez. **El artículo original no se abrió**: la ecuación se tomó de la implementación SENSE (`sense.surface.dubois95`, que cita a Ulaby 2014, cap. 10.6, ec. 10.41) y los límites de arXiv:2412.11874 | [V] (fuente secundaria) |
| R9 | Topp, G. C., Davis, J. L., Annan, A. P. (1980). «Electromagnetic determination of soil water content: measurements in coaxial transmission lines». *Water Resour. Res.* 16, 574–582 | ε = 3.03 + 9.3 m_v + 146.0 m_v² − 76.7 m_v³. **Original no abierto**: tomada de arXiv:1006.4307 | [V] (fuente secundaria) |
| R10 | Otsu, N. (1979). «A Threshold Selection Method from Gray-Level Histograms». *IEEE Trans. Syst. Man Cybern.* 9(1), 62–66. doi:10.1109/TSMC.1979.4310076 | Umbral de Otsu (el que usa el código del equipo). Metadatos vistos en resultados de búsqueda; **artículo no abierto** | [?] |
| R11 | NASA Earthdata, alerta «Permanent NISAR Instrument Data Gap» (28 ago 2026). https://www.earthdata.nasa.gov/data/alerts-outages/permanent-nisar-instrument-data-gap | Corte de datos de 2026-07-27 22:03:25 a 2026-08-10 00:55:27 UTC | [V] |
| R12 | NASA Earthdata, alerta «NISAR provisional data product production paused». https://www.earthdata.nasa.gov/data/alerts-outages/nisar-provisional-data-product-production-paused | Producción PROVISIONAL detenida el 2026-10-23 23:53 UTC; reanudación con atrasado después del 6 nov 2026 | [V] |
| R13 | WMO OSCAR, instrumento «SAR-L (NISAR)». https://space.oscar.wmo.int/instruments/view/sar_l_nisar | Ángulo de incidencia 33–47°, banda 1.215–1.3 GHz, modos de polarización | [V] |
| R14 | NASA CMR (Common Metadata Repository), consultas del 14 nov 2026. https://cmr.earthdata.nasa.gov/search/ | IDs de colección y escenas concretas (fechas, track, frame, enlaces HDF5) | [V] |
| R15 | Código del equipo `nsac-project-main` (`pipeline/fetch_nisar.py`, `README.md`) | Estructura del pipeline, sesión de Earthdata, lectura de la ventana, umbrales de clasificación | [V] (como código) |

**Noticias (todas abiertas el 14 nov 2026)**

| ID | Medio y fecha | Contenido usado |
|---|---|---|
| N1 | América TV, 23 oct 2026. https://www.americatv.com.pe/noticias/amp/actualidad/tumbes-lluvia-mas-ocho-horas-provoca-inundaciones-n523751 | Más de 8 h de lluvia en Tumbes; Zarumilla, Puerto Pizarro y Tumbes ciudad afectados; 52 mm en Puerto Pizarro según Senamhi |
| N2 | Caretas, 20 oct 2026. https://caretas.pe/?p=840084 | Lluvias desde la noche del 19 oct; ~30 cm de agua en el mercado de Chulucanas; también Tambogrande, Castilla, Sullana y Paita; alerta por quebradas en Ayabaca y Huancabamba |
| N3 | Caretas, 3 nov 2026. https://caretas.pe/?p=842604 | Desborde del río Zarumilla la madrugada del 2 nov en Puerto Perú (Aguas Verdes): más de 300 familias; El Tigre 1 021,33 m³/s; El Salto 104,1 mm/24 h |
| N4 | El Búho, 2 nov 2026. https://elbuho.pe/?p=441556 | Río Tumbes 1 021,33 m³/s (El Tigre, 05:00); desborde de la quebrada Marco Felipe (Zarumilla); Corrales y San Juan de la Virgen inundados; 70 % de la lluvia anual en menos de 30 días (aeropuerto) |
| N5 | Andina, 2 nov 2026. https://andina.pe/agencia/noticia-el-nino-tumbes-endures-extraordinary-rainfall-of-up-to-110-liters-per-square-meter-1098315.aspx | Zarumilla, la provincia más afectada; El Salto 110,3 mm el 2 nov y 102 mm el 27 oct; aviso rojo para Zarumilla, Sullana y parte de Ayabaca |
| N6 | El Peruano, 2 nov 2026. https://elperuano.pe/noticia/308191-fuerzas-armadas-activan-puente-terrestre-para-evitar-que-morropon-quede-aislada-por-el-nino | Quebrada Serrán activada en Salitral (Morropón); drenaje en Las Azucenas y en la av. Sánchez Cerro (Piura) |

> **Nota:** N3 da 104,1 mm/24 h para El Salto y N5 da 110,3 mm. Son dos cifras distintas para la misma estación y no las concilio aquí.

## 1. Configuración

Todo lo configurable está en `nisar_cn/config.py`. Cada constante lleva allí su fuente.

In [ ]:
MODO = "DEMO"          # "DEMO" (sintético, sin descargas) o "REAL" (CMR + streaming con Earthdata)

import sys
from pathlib import Path
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nisar_cn").is_dir())   # carpeta del proyecto
sys.path.insert(0, str(RAIZ))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nisar_cn import config, fisica, busqueda, lectura, indicadores, exportar, pipeline
CACHE, SALIDAS = RAIZ / "cache", RAIZ / "salidas"
print("Modo:", MODO, "| raíz del proyecto:", RAIZ)

### 1.1 Credenciales desde `.env`

El código busca un archivo `.env` en la carpeta del proyecto (`NISAR_ElNino_2026`) y hasta tres carpetas más arriba; si no lo encuentra, usa el de `Descargas\NassaSpaceAppChallenge`. Lo más limpio es copiar tu `.env` dentro de `NISAR_ElNino_2026`. Acepta una llave (`EARTHDATA_TOKEN=...` u otros nombres habituales, ver `nisar_cn/acceso.py`) o el par `EARTHDATA_USER` / `EARTHDATA_PASS`.

**Nunca se imprimen los valores, solo los nombres de las variables.** El `.gitignore` excluye `.env` para que la llave no termine en GitHub.

En modo REAL la celda siguiente también **prueba el acceso** leyendo los primeros 8 bytes de una escena real (24 oct 2026, Tumbes). No descarga el archivo de 6,8 GB. Si las credenciales sirven, esos bytes son la firma de un archivo HDF5.

> **Fuente** — Streaming con credenciales de Earthdata: ASF, *NISAR Data User Guide*, «Stream Data» · Estatus [V]. Escena y URL: R14 · Estatus [V].
> **[?]** La firma de 8 bytes `\x89HDF\r\n\x1a\n` viene de la especificación del formato HDF5 (The HDF Group, *HDF5 File Format Specification*, «Format Signature»), que no abrí en esta sesión. La comprobé en local con un archivo escrito por `h5py`.


In [ ]:
from nisar_cn import acceso
ruta_env, nombres = acceso.cargar_env()
print(".env:", ruta_env if ruta_env else "NO ENCONTRADO")
print("variables definidas (solo nombres):", nombres)
print("credencial que se usará:", acceso.credenciales())

URL_PRUEBA = ("https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_V1/"
              "NISAR_L2_PR_GCOV_034_003_A_176_4005_DHDH_A_20261024T110112_20261024T110149_P05036_N_F_J_001/"
              "NISAR_L2_PR_GCOV_034_003_A_176_4005_DHDH_A_20261024T110112_20261024T110149_P05036_N_F_J_001.h5")
if MODO == "REAL":
    try:
        ok, cab = acceso.probar_acceso(URL_PRUEBA)
        print("Acceso a NASA:", "OK — es un HDF5" if ok else f"respuesta inesperada {cab!r} (¿la llave expiró?)")
    except Exception as e:
        print("Acceso a NASA: FALLÓ →", type(e).__name__, str(e)[:300])
        print("Si es 401/403: entra una vez en https://search.asf.alaska.edu con tu cuenta y autoriza la aplicación ASF,"
              " o genera una llave nueva en https://urs.earthdata.nasa.gov (Generate Token).")
else:
    print("DEMO: no se prueba el acceso. Cambia MODO = 'REAL' en la primera celda.")

**Colecciones NISAR**

> **Fuente** — R14 (IDs leídos del CMR) · R3 (significado de cada madurez y precaución al mezclarlas) · R4 (GUNW) · R3 (SME2 sin versión VALIDATED aún) · Estatus [V]

In [ ]:
config.COLECCIONES

**Restricciones temporales del registro NISAR**

> **Fuente** — R11 (corte permanente de datos) · R12 (pausa de la producción PROVISIONAL desde el 23 oct 2026 23:53 UTC, con atrasado después del 6 nov) · R3 (VALIDATED desde el 21 oct 2026) · Estatus [V]
>
> La pausa de R12 explica por qué, al 14 nov 2026, todavía no había escenas de Tumbes posteriores al 2 nov en el CMR (R14).

In [ ]:
config.GAP_START, config.GAP_END, config.PAUSA_PR

**Zonas de estudio.** Cada zona está ligada a la noticia que la justifica. El punto (lat, lon) lo elegí yo para la consulta y la caja es de ±`R_DEG` grados alrededor.

> **Fuente** — N1–N6 (daños y lugares) · R14 (en los puntos de Tumbes, Zarumilla, Puerto Pizarro, Piura, Chulucanas, Salitral, Sullana y Ayabaca la consulta del 14 nov 2026 devolvió escenas GCOV) · Estatus de las coordenadas: **[A]**, puntos elegidos por mí y no tomados de una fuente.
>
> **[?] Hueco:** las coordenadas de los sectores que nombran las noticias (Puerto Perú, Las Azucenas, quebradas Marco Felipe y Serrán) deberían venir de una fuente cartográfica (IGN, INEI u OpenStreetMap). El equipo debe proporcionarla o confirmarla.

In [ ]:
pd.DataFrame([{"id": k, "nombre": z["nombre"], "región": z["region"], "noticias": z["noticias"],
               "lat": z["punto"][0], "lon": z["punto"][1]} for k, z in config.ZONAS.items()])

**Eventos.** Ventana de cada evento según la noticia (hora local Lima, UTC−5). El análisis buscará la última escena antes de `inicio` y la primera después de `fin`, en la misma geometría.

> **Fuente** — N2 (Piura: lluvias desde la noche del 19 oct) · N1 (Tumbes: 23 oct) · N5 (El Salto: 102 mm el 27 oct) · N3, N4 y N5 (1–2 nov) · N6 (Salitral, 2 nov) · Estatus [V]
>
> **[A]** Las horas de inicio y fin son una acotación mía de cada ventana a partir de las fechas que dan las noticias.

In [ ]:
pd.DataFrame([{**{k: v for k, v in e.items() if k != "zonas"}, "zonas": ", ".join(e["zonas"]),
               "noticia": config.NOTICIAS[e["fuente"]]} for e in config.EVENTOS])

## 2. Fundamento físico: rugosidad y constante dieléctrica

**Lo que dice el reto.** El texto de fondo del reto afirma que los datos de radar son más sensibles a la rugosidad y a la constante dieléctrica, y que la constante dieléctrica alta del agua resalta cuerpos de agua, inundaciones y humedad del suelo.
> **Fuente** — R1, sección *Background*, 3.er párrafo · Estatus [V]

**Mecanismos de dispersión y polarización**
- Superficie rugosa (suelo desnudo, agua): más sensible a VV.
- Volumen (hojas y ramas): más sensible a la polarización cruzada (HV, VH).
- Rebote doble (edificios, troncos, vegetación inundada): más sensible a HH.
> **Fuente** — R6, sección *Polarization and Scattering Mechanisms* · Estatus [V]

**El agua en calma da reflexión especular y retrodispersión baja; la tierra, más rugosa, da retrodispersión más alta. En terreno abierto σ⁰ sube con la humedad del suelo.**
> **Fuente** — R7, diapositivas 5, 6 y 10 · Estatus [V]

**Consecuencia para NISAR en esta zona.** Las escenas de Tumbes y Piura son `DHDH` (HH + HV) o `SHSH` (solo HH) (R14). El código `DH` significa HH/HV y `SH` solo HH (R2, convención `POLE`). **No hay VV**, que es el canal más sensible a la superficie rugosa (R6).
> **Fuente** — R2, R6, R14 · Estatus [V]

### 2.1 Modelo de Dubois para HH

$$
\sigma^0_{HH} = 10^{-2.75}\,\frac{\cos^{1.5}\theta}{\sin^{5}\theta}\;10^{\,0.028\,\varepsilon'\tan\theta}\;(ks\,\sin\theta)^{1.4}\;\lambda^{0.7}
$$

donde ε' es la parte real de la constante dieléctrica relativa, k = 2π/λ, s es la altura RMS de la rugosidad, θ el ángulo de incidencia y λ la longitud de onda **en cm**.

**Validez:** m_v < 0.35 cm³/cm³, ks < 3, θ > 30°.

> **Fuente** — R8 (Dubois et al., 1995). Ecuación copiada de la implementación SENSE `sense.surface.dubois95._hh`, que cita Ulaby (2014), cap. 10.6. Límites de validez de arXiv:2412.11874, ec. 2–8 · Estatus [V] (fuente secundaria; el artículo original no se abrió)

### 2.2 Relación de Topp (ε ↔ humedad volumétrica)

$$
\varepsilon(m_v) = 3.03 + 9.3\,m_v + 146.0\,m_v^2 - 76.7\,m_v^3
$$

> **Fuente** — R9 (Topp et al., 1980), tomada de arXiv:1006.4307, sección 5 · Estatus [V] (fuente secundaria)

### 2.3 Sensibilidades (deducción a partir de 2.1)

Al tomar 10·log₁₀ de la ecuación de Dubois:

$$
\frac{\partial \sigma^0_{HH,\mathrm{dB}}}{\partial \varepsilon'} = 10\cdot 0.028\,\tan\theta \;\;[\mathrm{dB}],
\qquad
\Delta\sigma^0_{HH,\mathrm{dB}}\big|_{\varepsilon,\theta\ \mathrm{fijos}} = 14\,\log_{10}\!\frac{(ks)_2}{(ks)_1}\;\;[\mathrm{dB}]
$$

Si entre dos fechas **la rugosidad no cambia** (misma ks), el cambio en dB solo depende de ε:

$$
\Delta\varepsilon' = \frac{\Delta\sigma^0_{HH,\mathrm{dB}}}{0.28\,\tan\theta}
$$

> **Fuente** — Derivación algebraica mía sobre la ecuación de R8 · Estatus **[A]**. El supuesto «ks constante entre fechas» es mío. Donde hubo depósito de lodo o cambio de cobertura (quebradas activadas), el supuesto falla, y la detección de cambios mezcla rugosidad y humedad.

**Por qué no se separan rugosidad y humedad con una sola fecha.** Con HH solamente hay una ecuación (2.1) para dos incógnitas (ε', ks). Dubois resuelve ambas con HH **y VV** (R8, ec. de σ⁰_VV), pero estas escenas no tienen VV (R2, R14). Por eso este notebook usa **dos fechas** con ks supuesta constante, y HV solo como indicador cualitativo de dispersión de volumen (R6).
> **Fuente** — R2, R6, R8, R14 + razonamiento mío · Estatus [A]

In [ ]:
lam = fisica.lambda_cm("4005")
print(f"λ banda A, modo 40+5 MHz: {lam:.2f} cm  (R5 + c/f)")
for th in (33, 40, 47):
    print(f"θ={th}°: ∂σdB/∂ε' = {fisica.dsigma_deps_db(np.radians(th)):.3f} dB   [A]")
print("Duplicar ks cambia σ⁰_HH en", round(float(fisica.dsigma_rugosidad_db(1, 2)), 2), "dB   [A]")

In [ ]:
mv = np.linspace(0.02, 0.34, 60)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for ks in (0.3, 0.6, 1.0, 2.0):
    for th, ls in ((33, "-"), (47, "--")):
        ax[0].plot(mv, 10*np.log10(fisica.dubois_hh(fisica.topp_eps(mv), ks, np.radians(th), lam)), ls, label=f"ks={ks}, θ={th}°")
ax[0].set_xlabel("m_v (cm³/cm³)"); ax[0].set_ylabel("σ⁰_HH (dB)"); ax[0].set_title(f"Dubois (R8) + Topp (R9), λ={lam:.1f} cm")
ax[0].legend(fontsize=7, ncol=2)
ks = np.linspace(0.1, 2.9, 60)
for m in (0.05, 0.15, 0.30):
    ax[1].plot(ks, 10*np.log10(fisica.dubois_hh(fisica.topp_eps(m), ks, np.radians(40), lam)), label=f"m_v={m}")
ax[1].set_xlabel("ks"); ax[1].set_ylabel("σ⁰_HH (dB)"); ax[1].set_title("θ = 40°"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Rango de θ (33–47°) según R13. Las curvas son el modelo, no una medición.")

## 3. Búsqueda de escenas

En modo REAL se consulta el CMR por zona, producto y madurez, ordenado por fecha. El nombre de cada escena se interpreta con la convención de R2: `NISAR_IL_PT_PROD_CYL_REL_P_FRM_MODE_POLE_S_Start_End_CRID_A_C_LOC_CTR`.

En modo DEMO se usa un catálogo **fijo** con escenas reales que leí del CMR el 14 nov 2026 (R14) para Tumbes, Zarumilla y Piura, más las fechas que encajan con los eventos.

Código: `nisar_cn/busqueda.py`.

> **Fuente** — R2 (convención de nombres) · R14 (catálogo) · R15 (la búsqueda del equipo usa `asf_search`; aquí se usa el CMR directamente para no depender de una sesión) · Estatus [V]

In [ ]:
CAT = pipeline.construir_catalogo(MODO)
pd.DataFrame([{"zona": z, "escenas": len(v),
               "geometrías": sorted({f"{g['REL']}_{g['P']}_{g['FRM']}" for g in v}),
               "desde": min((g["t"] for g in v), default=None), "hasta": max((g["t"] for g in v), default=None)}
              for z, v in CAT.items()])

### 3.1 Pares antes/después por evento

Para cada evento y zona se busca, **en cada geometría (track, dirección, frame) por separado**, la última escena antes del inicio y la primera después del fin. Se descartan las escenas dentro del corte de R11 y se exige el mismo modo de polarización en las dos fechas.

> **Fuente** — R15 (el equipo ya fija una sola geometría para que todos los cuadros sean comparables) · R11 · R3 (mezclar madurez puede producir diferencias de procesamiento: se registra en la columna `mezcla_madurez`) · Estatus [V]

In [ ]:
PARES = pipeline.tabla_pares(CAT)
PARES.drop(columns=["_a", "_d"])

## 4. Acceso a los datos reales por streaming (modo REAL)

Cada GCOV pesa entre 6 y 7 GB (por ejemplo, 6 819 938 304 bytes la escena del 24 oct 2026 de Tumbes, según su ficha CMR). Por eso **no se descargan completos**. La guía oficial indica que los productos NISAR se pueden leer por streaming con `fsspec` sobre HTTPS y credenciales de Earthdata, para tomar solo los bytes necesarios sin bajar el archivo entero.

Código: `nisar_cn/acceso.py`.

Credenciales: variables de entorno `EARTHDATA_USER` y `EARTHDATA_PASS`, como el `aiohttp.BasicAuth` del ejemplo oficial. Como alternativa se puede usar `EARTHDATA_TOKEN`, enviado como cabecera `Authorization: Bearer`, como en el código del equipo.

> **Fuente** — ASF, *NISAR Data User Guide*, sección «Stream Data» (ejemplo con `fsspec.filesystem('https', client_kwargs={'auth': aiohttp.BasicAuth(...)})`, `block_size=16 MB`, `cache_type='background'`) · tamaño del archivo: R14 · cabecera Bearer: R15 · Estatus [V]
>
> **[A]** El ejemplo oficial abre el archivo con `xarray` y el motor `h5netcdf`. Aquí el objeto de `fsspec` se pasa a `h5py.File`, que acepta objetos tipo archivo. Esto no lo probé contra un archivo real.

## 5. Lectura de la ventana de la zona

Se leen solo las filas y columnas de la caja, en la proyección nativa del producto (UTM, R2). Así el área de píxel es exacta (|dx·dy|) y no se reproyecta para el análisis.

Código: `nisar_cn/lectura.py`.

Capas que se leen (R2):
- `HHHH` y `HVHV`: γ⁰ en potencia, con corrección radiométrica de terreno.
- `rtcGammaToSigmaFactor`: factor para pasar de γ⁰ a σ⁰. El modelo de Dubois está escrito para σ⁰, así que se aplica antes de invertir ε.
- `numberOfLooks`: número de looks por píxel.
- `mask`: se lee y se guarda, pero **no se interpreta** (ver hueco abajo).

> **Fuente** — R2 (capas y significado) · R15 (`find_grid` y el recorte por ventana, adaptados) · Estatus [V]
>
> **[?] Hueco:** R2 dice que `mask` describe el conjunto de muestras promediadas, pero no da la tabla de valores en la parte que abrí. Para usarla como máscara de calidad hace falta la especificación JPL D-102274 Rev. F, sección 4.3 (Shiroma, 2026). El equipo debe revisarla.
>
> **Ángulo de incidencia:** la guía oficial (página «Metadata», tabla 1) indica que el GCOV trae `incidenceAngle` como cubo de metadatos 3D (alto × largo × ancho, por ejemplo de forma (20, 226, 68)), no como capa en `grids` · Estatus [V]. El código lo localiza por nombre. Interpolarlo a la grilla de la zona queda pendiente; mientras tanto se usa el punto medio del rango 33–47° de R13 y se avisa · **[?]**

### 5.1 Archivos sintéticos para el modo DEMO

Función `lectura.demo_h5`. Escribe HDF5 con la **misma estructura** que un GCOV (rutas de R2) y valores inventados para probar el código: tierra, un cauce y una llanura que «se inunda» después del evento. **No representan la realidad.**

## 6. Indicadores por escena y por par

Código: `nisar_cn/indicadores.py`.

| # | Indicador | Cálculo | Fuente |
|---|---|---|---|
| 1 | Agua abierta | HH (dB) < umbral de Otsu, acotado a [−22, −14] dB | Otsu: R10 [?] · física: R7 [V] · **acotación: R15 [?]** |
| 2 | Vegetación inundada | HH > −3.5 dB y HH − HV > 7 dB | Física: R6, R7 [V] · **umbrales: R15 [?]** |
| 3 | Cambio de agua | agua_después ∧ ¬agua_antes (ganada) y viceversa (perdida) | R15 [V] |
| 4 | Δσ⁰_HH (dB) | 10·log₁₀(σ⁰₂/σ⁰₁) en píxeles de tierra estable (no agua en ninguna fecha) | Definición; σ⁰ = γ⁰·factor (R2) [V] |
| 5 | Δε' (constante dieléctrica) | Δσ⁰_dB / (0.28·tan θ), suponiendo ks constante | Sección 2.3 [A] |
| 6 | HV/HH | cociente en dB (HV_dB − HH_dB) | Indicador de volumen frente a superficie: R6 [V]; la lectura cuantitativa es [A] |
| 7 | Coherencia | capa de coherencia del GUNW del par de 12 días | R4 [V] |
| 8 | Humedad del suelo | capas SME2 (DSG, PMI, TSR) | R3 [V] |

> **[?] Hueco — umbrales en dB.** −22, −14, −3.5 y 7 dB vienen del código del equipo (R15, `classify`) y el README no cita su origen. No encontré una referencia que los respalde para banda L: R7 no da umbrales en dB. **Pido al equipo la referencia** (autor, título, año, tabla o figura) o, si no la hay, presentarlos como parámetros calibrados a ojo sobre el Ucayali.
>
> **[?] Hueco — viento sobre el agua.** El agua agitada por viento o lluvia podría verse más brillante y escaparse del umbral. R7 no lo trata y no tengo una fuente abierta para afirmarlo. Pido referencia antes de usarlo en la presentación.
>
> **Filtro de speckle:** mediana 3×3 como en R15. No tiene referencia bibliográfica en el código del equipo: **[?]**.

In [ ]:
_, _, TABLA, MAPAS = pipeline.ejecutar(MODO, CACHE)
if MODO == "DEMO":
    print("⚠ MODO DEMO: valores de archivos SINTÉTICOS. No son resultados.")
cols = ["evento", "zona", "geom", "antes", "despues", "mezcla_madurez", "agua_antes_km2", "agua_despues_km2",
        "agua_ganada_km2", "dsigma_tierra_med_db", "deps_tierra_med", "hvhh_antes_db", "hvhh_despues_db", "cobertura", "theta"]
TABLA[cols].round({c: 3 for c in cols if c not in ("antes", "despues", "theta", "evento", "zona", "geom", "mezcla_madurez")}) if len(TABLA) else "Sin pares analizables"

In [ ]:
if len(TABLA):
    r, m = TABLA.iloc[0], MAPAS[0]
    res, dsig, deps, tierra = m["res"], m["dsig"], m["deps"], m["tierra"]
    cmap = plt.matplotlib.colors.ListedColormap(["#d9d2c3", "#2f8cff", "#2fd1a0"])
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    for i, k in enumerate(("a", "d")):
        ax[i].imshow(np.where(res[k]["cls"] == 255, np.nan, res[k]["cls"]), cmap=cmap, vmin=0, vmax=2)
        ax[i].set_title(("Antes " if k == "a" else "Después ") + str(r["antes" if k == "a" else "despues"])[:10])
    im = ax[2].imshow(np.where(tierra, dsig, np.nan), cmap="RdBu", vmin=-4, vmax=4); ax[2].set_title("Δσ⁰_HH (dB), tierra estable"); plt.colorbar(im, ax=ax[2])
    im = ax[3].imshow(np.where(tierra, deps, np.nan), cmap="BrBG", vmin=-15, vmax=15); ax[3].set_title("Δε' (ks constante) [A]"); plt.colorbar(im, ax=ax[3])
    for a_ in ax: a_.axis("off")
    plt.suptitle(f"{r['evento']} · {config.ZONAS[r['zona']]['nombre']} · {r['geom']}" + ("  —  DEMO SINTÉTICO" if MODO == "DEMO" else ""))
    plt.tight_layout(); plt.show()

## 7. Coherencia (GUNW) y humedad del suelo (SME2)

- **GUNW** trae la magnitud de coherencia ya calculada para pares co-pol de 12 días, a 20 y 80 m (R4). Una caída de coherencia entre fechas marca cambio de superficie: agua, lodo o daño.
- **SME2** trae tres estimaciones de humedad (DSG, PMI y TSR) a 200 m en la malla EASE 2.0 (R3).

Las rutas internas de esas capas **no aparecen en las páginas que abrí**. Por eso el código las busca por nombre dentro del archivo y las lista antes de usarlas.

> **Fuente** — R3, R4 · Estatus [V]. Además, la guía oficial (página «Product Known Issues») recomienda usar la capa de humedad base **`/science/LSAR/SME2/grids/soilMoisture`** y advierte que los indicadores de superficie y de recuperación no están completamente implementados, lo que deja valores nulos en esa capa · Estatus [V].
>
> **[?] Hueco:** ruta exacta de la coherencia dentro del GUNW. La sección siguiente la busca por nombre.

In [ ]:
if MODO == "REAL":
    from nisar_cn.acceso import abrir_remoto
    import json
    for z in ("zarumilla", "piura"):
        lat, lon = config.ZONAS[z]["punto"]
        for prod in ("GUNW", "SME2"):
            gs = busqueda.cmr_buscar(config.COLECCIONES[prod]["PROVISIONAL"], lat, lon, temporal="2026-10-01T00:00:00Z,2026-11-14T23:59:59Z")
            if gs:
                print(z, prod, gs[-1]["nombre"])
                print(json.dumps(lectura.explorar_capas(abrir_remoto(gs[-1])), indent=1)[:1500])
else:
    print("DEMO: se omite (requiere archivos GUNW/SME2 reales). En REAL lista las capas para fijar las rutas.")

## 8. Tabla para el storytelling y manifiesto para la web

Código: `nisar_cn/exportar.py`. Une cada resultado con su evento, su titular y el enlace a la noticia. El manifiesto mantiene la forma `window.NISAR_MANIFEST` que lee `app.js` del equipo y agrega los campos `zona` y `evento`.

Además escribe las capas PNG de cada par (`nisar_cn/mapas.py`) y abre el mapa `2_mapa/MAPA_NISAR.html`, que muestra todas las zonas, los eventos con su noticia y, para cada par, SAR antes/después, agua, cambio, Δσ⁰ y Δε′.

> **Fuente** — R15 (formato del manifiesto) · N1–N6 · Estatus [V]

También procesa la **serie temporal** (`nisar_cn/serie.py`): todas las fechas NISAR disponibles por zona y órbita, cada una comparada con la primera fecha. Escribe las capas para la barra de tiempo del mapa y un **GIF animado** por serie en `salidas/serie_<modo>/`.

Para contrastar, agrega la **lluvia diaria** de cada zona (Open-Meteo, reanálisis ECMWF; `nisar_cn/lluvia.py`), la **humedad del suelo NISAR L3 SME2** y la **coherencia NISAR L2 GUNW** (`nisar_cn/extra.py`), y la **crónica de noticias y reportes oficiales** fechados (`config.CRONICA`).

In [ ]:
if len(TABLA):
    STORY = exportar.storytelling(TABLA, MODO)
    CAPAS = exportar.capas_mapa(TABLA, MAPAS, MODO, SALIDAS)          # PNG por par → salidas/mapas_<modo>/
    from nisar_cn import serie
    SERIE = serie.serie_temporal(CAT, MODO, CACHE, SALIDAS)            # todas las fechas + GIF por zona y órbita
    from nisar_cn import lluvia, extra
    LLUVIA = lluvia.lluvia_todas(CACHE)                                 # lluvia diaria por zona (Open-Meteo, reanálisis)
    EXTRA = extra.productos_extra(sorted({s["zona"] for s in SERIE}), CACHE, SALIDAS, MODO)   # SME2 + GUNW
    csv, js = exportar.escribir(STORY, MODO, SALIDAS, CAPAS, SERIE, LLUVIA, EXTRA)
    display(STORY); print("Escrito:", csv, js, f"y {len(CAPAS)} juegos de capas PNG")
    VISOR = (RAIZ / "2_mapa" / "MAPA_NISAR.html").resolve()
    print("Mapa:", VISOR)
    import webbrowser; webbrowser.open(VISOR.as_uri())                # abre el mapa en tu navegador

## 9. Pendientes y huecos (para cerrar antes de la presentación)

1. **[?] Referencia de los umbrales en dB** (−22, −14, −3.5 y 7 dB) del código del equipo (R15).
2. **[?] Valores de la capa `mask`** del GCOV: especificación JPL D-102274 Rev. F, sección 4.3 (Shiroma, 2026).
3. **[?] Interpolar el cubo `incidenceAngle`** a la grilla de la zona. Su existencia está verificada en la guía, página «Metadata». Mientras tanto se usan 40°.
4. **[?] Ruta de la coherencia (GUNW)**: ejecutar la sección 7 en modo REAL y fijarla. La de humedad (`/science/LSAR/SME2/grids/soilMoisture`) ya está verificada.
5. **[?] Coordenadas de los sectores** que nombran las noticias, desde una fuente cartográfica.
6. **[?] Efecto del viento sobre la superficie del agua**: falta referencia.
7. **[?] Otsu (R10)**: abrir el artículo original para pasar a [V].
8. **R8 y R9 en fuente secundaria**: conviene abrir Dubois et al. (1995) y Topp et al. (1980) originales.
9. **Escenas posteriores al 2 nov** en Tumbes: la producción PROVISIONAL estuvo pausada hasta el 6 nov (R12). Hay que volver a correr la sección 3 en modo REAL.
10. **Mezcla de madurez** PROVISIONAL → VALIDATED en todos los pares de octubre (R3). Antes de atribuir el Δσ⁰ al terreno, comparar zonas estables (roca, desierto sin lluvia). Esa comparación es una propuesta mía **[A]** sin referencia.